# 01 — Financial Statement Spreading & Earnings Normalisation

## Purpose
This notebook demonstrates the **first step** in any bank credit analysis: converting raw financial statements into a standardised spread template, then normalising earnings to EBITDAO.

**Bank context:** Before a credit analyst can calculate ratios or score a borrower, they must first *spread* the financial statements — organising the Income Statement, Balance Sheet, and Cash Flow Statement into a consistent format across multiple periods (typically FY-2, FY-1, FY0).

**Portfolio context:**
- bank credit framework: Steps 1-4 (Revenue Factors → Gross Margin → Normalised Earnings → Total Available for Servicing)
- Excel model: `Company_Inputs` sheet

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_sme_dataset
from src.financial_spreading import spread_borrower, format_spread_display, PERIOD_ORDER
from src.normalisation import normalise_earnings, total_available_for_servicing

pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

# Load dataset
df = generate_sme_dataset(n_borrowers=80, seed=42)
print(f'Dataset: {df["borrower_id"].nunique()} borrowers, {len(df)} rows')
print(f'Industries: {df["anzsic_division"].unique().tolist()}')

## Step 1: Financial Statement Spread — Base Case Borrower

We start with **Base Case AU SME Pty Ltd** (borrower_id=0), which matches the Excel model exactly. This allows us to verify all downstream calculations.

In [ ]:
# Spread the base case borrower
spread = spread_borrower(df, borrower_id=0)
display_spread = format_spread_display(spread, 'Base Case AU SME Pty Ltd')

print('='*70)
print('FINANCIAL STATEMENT SPREAD — Base Case AU SME Pty Ltd')
print('Periods: FY-2 (Audited) | FY-1 (Audited) | FY0 (Management Accounts)')
print('='*70)
display(display_spread)

## Step 2: Earnings Normalisation (EBITDAO)

Banks normalise earnings by:
1. Starting with **EBIT** (Operating Profit)
2. Adding back **Depreciation & Amortisation** and **Interest** → EBITDA
3. Adjusting for **Owner Salary** (the 'O' in EBITDAO) — if the owner takes below-market salary, EBITDAO is reduced
4. Removing **one-off income** (e.g. government grants) and adding back **one-off expenses**

The result is **EBITDAO** — the true operating cash generation of the business.

In [ ]:
# Normalisation waterfall — demonstrate with owner salary adjustment
waterfall = normalise_earnings(
    spread,
    owner_salary_adjustment=-50_000,  # Owner takes $100k but market rate is $150k
    one_off_income=0,
    one_off_expense=0,
)

print('='*70)
print('EARNINGS NORMALISATION WATERFALL')
print('Owner salary adjustment: -$50,000 (market rate $150k, actual $100k)')
print('='*70)

# Format for display
wf_display = waterfall.copy()
for col in PERIOD_ORDER:
    wf_display[col] = wf_display[col].apply(lambda x: f'${x:,.0f}' if pd.notna(x) else '')
display(wf_display)

## Step 3: Total Available for Servicing

The **"first way out"** test — can the business cover its debt obligations from operating cash flow?

This is the most critical test in commercial lending. A positive surplus means the business can service its debt from normal operations.

In [ ]:
# Use EBITDA as EBITDAO proxy (no owner adjustment for synthetic data)
ebitdao = spread.loc['ebitda'].values.astype(float)
interest = spread.loc['interest_expense'].values.astype(float)
principal = df[df['borrower_id']==0].sort_values('period')['scheduled_principal'].values.astype(float)

servicing = total_available_for_servicing(ebitdao, interest, principal)

print('='*70)
print('TOTAL AVAILABLE FOR SERVICING — First Way Out Test')
print('='*70)

serv_display = servicing.copy()
for col in ['EBITDAO', 'Interest', 'Scheduled Principal', 'Total Repayments', 'Surplus / (Deficit)']:
    serv_display[col] = serv_display[col].apply(lambda x: f'${x:,.0f}')
serv_display['DSCR (EBITDAO / Repayments)'] = servicing['DSCR (EBITDAO / Repayments)'].apply(lambda x: f'{x:.2f}x')
display(serv_display)

## Step 4: Revenue & Margin Trends (Visual)

Before diving into detailed ratio analysis, a credit analyst will visually assess revenue and margin trends.

In [ ]:
ref = df[df['borrower_id'] == 0].sort_values('period')
periods = ['FY-2', 'FY-1', 'FY0']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Revenue
axes[0].bar(periods, ref['revenue'] / 1e6, color='#2196F3', alpha=0.8)
axes[0].set_title('Revenue (A$M)', fontweight='bold')
axes[0].set_ylabel('A$ Millions')
for i, v in enumerate(ref['revenue'] / 1e6):
    axes[0].text(i, v + 0.2, f'${v:.1f}M', ha='center', fontsize=10)

# EBITDA & EBIT
x = np.arange(3)
w = 0.35
axes[1].bar(x - w/2, ref['ebitda'] / 1e6, w, label='EBITDA', color='#4CAF50', alpha=0.8)
axes[1].bar(x + w/2, ref['ebit'] / 1e6, w, label='EBIT', color='#FF9800', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(periods)
axes[1].set_title('EBITDA vs EBIT (A$M)', fontweight='bold')
axes[1].legend()

# Margins
axes[2].plot(periods, ref['ebitda'] / ref['revenue'] * 100, 'o-', label='EBITDA Margin', linewidth=2)
axes[2].plot(periods, ref['npat'] / ref['revenue'] * 100, 's-', label='Net Margin', linewidth=2)
axes[2].set_title('Margin Trends (%)', fontweight='bold')
axes[2].set_ylabel('%')
axes[2].legend()

fig.suptitle('Base Case AU SME Pty Ltd — Financial Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/01_revenue_margin_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5: Portfolio Overview — Spread Multiple Borrowers

In practice, a credit analyst would spread each borrower individually. Here we show the dataset composition.

In [ ]:
fy0 = df[df['period'] == 'FY0']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Industry distribution
industry_counts = fy0['anzsic_division'].value_counts()
axes[0].barh(industry_counts.index, industry_counts.values, color='#2196F3', alpha=0.8)
axes[0].set_title('Portfolio by Industry (ANZSIC)', fontweight='bold')
axes[0].set_xlabel('Number of Borrowers')

# Revenue distribution
axes[1].hist(fy0['revenue'] / 1e6, bins=20, color='#4CAF50', alpha=0.8, edgecolor='white')
axes[1].set_title('FY0 Revenue Distribution', fontweight='bold')
axes[1].set_xlabel('Revenue (A$M)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(fy0['revenue'].median() / 1e6, color='red', linestyle='--', label=f'Median: ${fy0["revenue"].median()/1e6:.1f}M')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/01_portfolio_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nPortfolio Summary:')
print(f'  Total borrowers: {fy0["borrower_id"].nunique()}')
print(f'  Revenue range: ${fy0["revenue"].min()/1e6:.1f}M to ${fy0["revenue"].max()/1e6:.1f}M')
print(f'  Median revenue: ${fy0["revenue"].median()/1e6:.1f}M')

---

## Key Takeaways for Credit Analysts

1. **Spreading is foundational** — standardised format enables consistent ratio calculation and peer comparison
2. **Normalisation matters** — EBITDAO removes owner-specific and one-off items to reveal true operating performance
3. **Three periods minimum** — banks require FY-2 (audited), FY-1 (audited), FY0 (management accounts) to assess trends
4. **Total Available for Servicing** is the critical "first way out" test — surplus after all debt commitments

**Next:** [02_Ratio_Analysis_and_Trends.ipynb](02_Ratio_Analysis_and_Trends.ipynb) — calculate 20+ credit ratios and assess 3-period trends.